In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

X_full = np.load('../data/X_train.npy')
y_full = np.load('../data/y_train.npy')
obs_count_full = np.load('../data/obs_count.npy')
obs_sse_full = np.load('../data/obs_sse.npy')

# Writeout

In [ ]:
n = X_full.shape[0]
n_train = (n * 4) // 5

np.random.seed(305)
shuffled_idx = np.random.permutation(range(n))
train_idx = shuffled_idx[:n_train]
test_idx = shuffled_idx[n_train:]

In [ ]:
np.save('X_train.npy', X_train:=X_full[train_idx])
np.save('y_train.npy', y_train:=y_full[train_idx])
np.save('obs_count.npy', obs_count_full[train_idx])
np.save('obs_sse.npy', obs_sse_full[train_idx])

np.save('X_test.npy', X_test:=X_full[test_idx])
np.save('y_test.npy', y_test:=y_full[test_idx])

In [ ]:

# Active GP tuning grid is generated by ../setup_nested_geo_eval.py.
gp_stage1_grid = pd.read_csv("gp_geo_stage1_grid.csv")
gp_stage1_grid.head()


In [ ]:
gp_stage1_grid.shape


# KNN Baseline

In [ ]:



class WifiKNNFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, indoor_scale=1.0, ap_scale=1.0, n_ap=None):
        self.indoor_scale = indoor_scale
        self.ap_scale = ap_scale
        self.n_ap = n_ap

    def fit(self, X, y=None):
        X = np.asarray(X)
        if self.n_ap is None:
            self.n_ap_ = int(np.nanmax(X[:, 4])) + 1
        else:
            self.n_ap_ = int(self.n_ap)
        return self

    def transform(self, X):
        X = np.asarray(X)
        transformed = X[:, [0, 1, 2]].astype(float).copy()
        transformed[:, 2] = transformed[:, 2] / self.indoor_scale

        ap_idx = X[:, 4].astype(int)
        ap_one_hot = np.zeros((X.shape[0], self.n_ap_), dtype=transformed.dtype)
        valid_ap = (ap_idx >= 0) & (ap_idx < self.n_ap_)
        ap_one_hot[np.flatnonzero(valid_ap), ap_idx[valid_ap]] = 1.0
        ap_one_hot = ap_one_hot / self.ap_scale

        return np.concatenate([transformed, ap_one_hot], axis=1)

In [ ]:
X_geo_tr = np.load('../data/geo_X_train.npy')
y_geo_tr = np.load('../data/geo_y_train.npy')
X_geo_val = np.load('../data/geo_X_val.npy')
y_geo_val = np.load('../data/geo_y_val.npy')

In [ ]:
np.arange(3, 21, 3)

In [ ]:
n_ap = int(np.nanmax(X_full[:, 4])) + 1

param_grid = {
    # "features__indoor_scale": [0.25, 0.5, 1.0, 2.0, 4.0],
    "features__indoor_scale": [0.5, 0.1, 2,],
    # "features__ap_scale": [0.5, 1.0, 2.0, 4.0, 8.0, 1e6],
    "features__ap_scale": [0.25, 0.5, 1.0,],
    # "knn__n_neighbors": np.arange(1, 21),
    "knn__n_neighbors": np.arange(3, 24, 3),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]  # Manhattan vs Euclidean
}

knn_pipeline = Pipeline([
    ('features', WifiKNNFeatureTransformer(n_ap=n_ap)),
    ('knn', KNeighborsRegressor()),
])

knn_grid = GridSearchCV(
    knn_pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",  # or "r2"
    n_jobs=-1,   # use all CPU cores
    verbose=0
)

knn_grid.fit(X_train, y_train)
print(knn_grid.best_params_)

In [ ]:
knn_grid.best_score_

In [ ]:
preds = knn_grid.predict(X_test)
sq_errs = (preds - y_test)**2
mse = sq_errs.mean()

plt.hist(sq_errs)
plt.title(f"MSE: {mse:.4f}")
plt.show()

# Cross validation

In [ ]:
CV_SPLIT_TYPES = ['rand', 'geo']
CV_N_SPLITS = 5
KNN_CV_RESULTS_PATH = 'knn_cv_mse_results.csv'


def fit_evaluate_knn_cv_split(split_type, split_idx):
    split_dir = f'../cv/{split_type}'

    X_cv_train = np.load(f'{split_dir}/X_train_{split_idx}.npy')
    y_cv_train = np.load(f'{split_dir}/y_train_{split_idx}.npy')
    X_cv_test = np.load(f'{split_dir}/X_test_{split_idx}.npy')
    y_cv_test = np.load(f'{split_dir}/y_test_{split_idx}.npy')

    knn_grid = GridSearchCV(
        knn_pipeline,
        param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=0,
    )
    knn_grid.fit(X_cv_train, y_cv_train)

    preds = knn_grid.predict(X_cv_test)
    mse = np.mean((preds - y_cv_test) ** 2)

    best_params = knn_grid.best_params_
    return {
        'split_type': split_type,
        'split_idx': split_idx,
        'n_train': X_cv_train.shape[0],
        'n_test': X_cv_test.shape[0],
        'mse': mse,
        'inner_cv_mse': -knn_grid.best_score_,
        'indoor_scale': best_params['features__indoor_scale'],
        'ap_scale': best_params['features__ap_scale'],
        'n_neighbors': best_params['knn__n_neighbors'],
        'weights': best_params['knn__weights'],
        'p': best_params['knn__p'],
    }


knn_cv_results = []
for split_type in CV_SPLIT_TYPES:
    for split_idx in range(CV_N_SPLITS):
        print(f'Fitting KNN {split_type} split {split_idx}')
        result = fit_evaluate_knn_cv_split(split_type, split_idx)
        knn_cv_results.append(result)
        print(
            f"{split_type} split {split_idx}: "
            f"MSE={result['mse']:.4f}, "
            f"indoor_scale={result['indoor_scale']}, "
            f"ap_scale={result['ap_scale']}, "
            f"n_neighbors={result['n_neighbors']}, "
            f"weights={result['weights']}, p={result['p']}"
        )

knn_cv_results_df = pd.DataFrame(knn_cv_results)
knn_cv_results_df.to_csv(KNN_CV_RESULTS_PATH, index=False)
knn_cv_results_df

# Full-data KNN prediction surface

In [ ]:
import dill
import sys, os
sys.path.append(os.getcwd())
sys.path.append(f"{os.getcwd()}/..")

import wifiplotting

full_knn_grid = GridSearchCV(
    knn_pipeline,
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=0,
)
full_knn_grid.fit(X_full, y_full)
print(full_knn_grid.best_params_)

X_grid = np.load('../data/X_test.npy')
grid_preds = full_knn_grid.predict(X_grid)

with open('../data/osm_context.pkl', 'rb') as f:
    osm_context = dill.load(f)

wlon_train, wlat_train = np.load('../data/world_train.npy').T
wlon_test, wlat_test = np.load('../data/world_test.npy').T

vmax = np.quantile(y_full, 0.975)
vmin = np.quantile(y_full, 0.025)

base_fig, base_ax, osm_metadata = osm_context.generate_base_axis(draw_buildings=False)

sc = base_ax.scatter(
    wlon_test,
    wlat_test,
    c=grid_preds,
    cmap='RdYlGn',
    vmin=vmin,
    vmax=vmax,
)

tr = base_ax.scatter(
    wlon_train,
    wlat_train,
    c=y_full,
    cmap='RdYlGn',
    vmin=vmin,
    vmax=vmax,
    alpha=1,
)
plt.ticklabel_format(style='plain', axis='both', useOffset=False)
plt.colorbar(tr)
plt.tight_layout()

# Geo fold 2 KNN prediction surface

In [ ]:
import dill
import sys, os
sys.path.append(os.getcwd())
sys.path.append(f"{os.getcwd()}/..")

import wifiplotting

GEO_VIS_SPLIT_IDX = 2
GEO_VIS_LAT_BINS = 20
GEO_VIS_LON_BINS = 20

geo_fold_dir = "../cv/geo"
X_geo_fold_train = np.load(f"{geo_fold_dir}/X_train_{GEO_VIS_SPLIT_IDX}.npy")
y_geo_fold_train = np.load(f"{geo_fold_dir}/y_train_{GEO_VIS_SPLIT_IDX}.npy")
X_geo_fold_test = np.load(f"{geo_fold_dir}/X_test_{GEO_VIS_SPLIT_IDX}.npy")
y_geo_fold_test = np.load(f"{geo_fold_dir}/y_test_{GEO_VIS_SPLIT_IDX}.npy")

geo_fold_knn_grid = GridSearchCV(
    knn_pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=0,
)
geo_fold_knn_grid.fit(X_geo_fold_train, y_geo_fold_train)
print(geo_fold_knn_grid.best_params_)

X_geo_fold_grid = np.load("../data/X_test.npy")
geo_fold_grid_pred = geo_fold_knn_grid.predict(X_geo_fold_grid)


In [ ]:
with open("../data/osm_context.pkl", "rb") as f:
    geo_fold_context = dill.load(f)

geo_fold_coord_all = np.load("../data/coord_train.npy")
geo_fold_y_all = np.load("../data/y_train.npy")
geo_fold_train_idx = np.load(f"{geo_fold_dir}/train_idx_{GEO_VIS_SPLIT_IDX}.npy")
geo_fold_test_idx = np.load(f"{geo_fold_dir}/test_idx_{GEO_VIS_SPLIT_IDX}.npy")
geo_fold_world_train = np.load(f"{geo_fold_dir}/world_train_{GEO_VIS_SPLIT_IDX}.npy")
geo_fold_world_test = np.load(f"{geo_fold_dir}/world_test_{GEO_VIS_SPLIT_IDX}.npy")
geo_fold_world_grid = np.load("../data/world_test.npy")

geo_fold_lat_edges, geo_fold_lon_edges, geo_fold_lat_grid, geo_fold_lon_grid, geo_fold_in_grid = wifiplotting.geo_vis_grid_from_lonlat(
    geo_fold_coord_all[:, 0],
    geo_fold_coord_all[:, 1],
    GEO_VIS_LAT_BINS,
    GEO_VIS_LON_BINS,
)
geo_fold_heldout_cells = wifiplotting.geo_vis_cells_from_indices(
    geo_fold_test_idx,
    geo_fold_lat_grid,
    geo_fold_lon_grid,
    geo_fold_in_grid,
)

geo_fold_vmin = np.quantile(geo_fold_y_all, 0.025)
geo_fold_vmax = np.quantile(geo_fold_y_all, 0.975)

fig, ax, surface = wifiplotting.plot_geo_fold_surface(
    geo_fold_context,
    geo_fold_world_grid,
    geo_fold_grid_pred,
    world_train=geo_fold_world_train,
    y_train=y_geo_fold_train,
    world_test=geo_fold_world_test,
    y_test=y_geo_fold_test,
    lat_edges=geo_fold_lat_edges,
    lon_edges=geo_fold_lon_edges,
    heldout_cells=geo_fold_heldout_cells,
    vmin=geo_fold_vmin,
    vmax=geo_fold_vmax,
    model_label="KNN",
    split_idx=GEO_VIS_SPLIT_IDX,
    surface_size=36,
    surface_alpha=0.7,
    show_train=False,
    test_size=72,
    cell_style={"edgecolor": "black"},
    show_colorbar=False,
)


# Geo fold 2 KNN error diagnostics

In [ ]:
geo_fold_test_pred = geo_fold_knn_grid.predict(X_geo_fold_test)
geo_fold_test_resid, geo_fold_test_sq_err, geo_fold_test_mse, top_error_local_idx, geo_fold_top_errors = wifiplotting.compute_heldout_error_summary(
    y_geo_fold_test,
    geo_fold_test_pred,
    test_idx=geo_fold_test_idx,
    coord_all=geo_fold_coord_all,
    X_test=X_geo_fold_test,
    top_n=4,
)
top_error_n = len(top_error_local_idx)
top_error_global_idx = geo_fold_test_idx[top_error_local_idx]

fig, ax = wifiplotting.plot_heldout_error_hist(
    geo_fold_test_sq_err,
    geo_fold_test_mse,
    top_error_local_idx,
    split_idx=GEO_VIS_SPLIT_IDX,
    model_label="KNN",
    top_n=top_error_n,
)

fig, ax, top_scatter = wifiplotting.plot_top_heldout_errors_map(
    geo_fold_context,
    geo_fold_world_test,
    geo_fold_test_sq_err,
    top_error_local_idx,
    lat_edges=geo_fold_lat_edges,
    lon_edges=geo_fold_lon_edges,
    heldout_cells=geo_fold_heldout_cells,
    split_idx=GEO_VIS_SPLIT_IDX,
    model_label="KNN",
    top_n=top_error_n,
    cell_style={"edgecolor": "black"},
)

geo_fold_top_errors
